The goal of this file is to Solve the SDP problem 

Feasible

Subject to: 
Pr(X - Y) = A_G
tr(Y) <= k 

Fix 1's on the diagonal of the matrix and then solve for frasability using submatrices. 

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))



In [ ]:
from utils.clique import generate_AG
from utils.clique import generate_AG_bad
# Import packages.
import cvxpy as cp
import numpy as np


In [ ]:
# single square graph
maximal_cliques = [
    [0,1],
    [1,2],
    [2,3],
    [3,0]
]
A_G = (generate_AG(4, maximal_cliques))
print(A_G)
n = A_G.shape[0]



for k in range(1, n + 1):
    Y = cp.Variable((n,n), symmetric=True)
    constraints = [Y >> 0, cp.trace(Y) <= k]
    # Impose only known entries

    for i in range(n):
        for j in range(n):
            if A_G[i,j] != 0:
                constraints.append(Y[i,j] == A_G[i,j])

    prob = cp.Problem(cp.Minimize(0), constraints)
    prob.solve()
    if(prob.status == "optimal"):
        
        print("Status:", prob.status)
        print("Solution Y:\n", Y.value)

        # Check rank
        eigvals = np.linalg.eigvalsh(Y.value)
        print("Eigenvalues:", eigvals)
        print("Numerical rank:", np.sum(eigvals > 1e-6))
        break

In [ ]:
# large cycle graph
k = 10
maximal_cliques = [[i, (i + 1) % k] for i in range(k)]

print(maximal_cliques)
A_G = (generate_AG(k, maximal_cliques))

n = A_G.shape[0]


for k in range(1, n + 1):
    Y = cp.Variable((n,n), symmetric=True)
    constraints = [Y >> 0, cp.trace(Y) <= k]
    # Impose only known entries

    for i in range(n):
        for j in range(n):
            if A_G[i,j] != 0:
                constraints.append(Y[i,j] == A_G[i,j])

    prob = cp.Problem(cp.Minimize(0), constraints)
    prob.solve()
    if(prob.status == "optimal"):
        
        print("Status:", prob.status)
        print("Solution Y:\n", Y.value)

        # Check rank
        eigvals = np.linalg.eigvalsh(Y.value)
        print("Eigenvalues:", eigvals)
        print("Numerical rank:", np.sum(eigvals > 1e-6))
        break

In [ ]:
# large cycle graph with one negative edge (no solution, k=1)
k = 10
maximal_cliques = [[i, (i + 1) % k] for i in range(k)]

print(maximal_cliques)
A_G = (generate_AG_bad(k, maximal_cliques, bad_edges=[[0,1]]))

n = A_G.shape[0]


for k in range(1, n + 1):
    Y = cp.Variable((n,n), symmetric=True)
    constraints = [Y >> 0, cp.trace(Y) <= k]
    # Impose only known entries

    for i in range(n):
        for j in range(n):
            if A_G[i,j] != 0:
                constraints.append(Y[i,j] == A_G[i,j])

    prob = cp.Problem(cp.Minimize(0), constraints)
    prob.solve()
    if(prob.status == "optimal"):
        
        print("Status:", prob.status)
        print("Solution Y:\n", Y.value)

        # Check rank
        eigvals = np.linalg.eigvalsh(Y.value)
        print("Eigenvalues:", eigvals)
        print("Numerical rank:", np.sum(eigvals > 1e-6))
        break

print("Status:", prob.status)

In [ ]:
import cvxpy as cp
import numpy as np
from itertools import combinations
# large cycle graph with one negative edge (no solution, k=1)
k = 20
maximal_cliques = [[i, (i + 1) % k] for i in range(k)]

print(maximal_cliques)
A_G = (generate_AG_bad(k, maximal_cliques, bad_edges=[[0,1]]))

n = A_G.shape[0]
k = 1  # number of allowed negative eigenvalues; completion number is the smallest k
num_trials = 100  # number of random subspaces to try
print(A_G)
best_result = None
best_obj = np.inf
eps = 1e-4  # small shift to push near-zero eigenvalues positive
neg_eigenvalues = []
'''
New program to attempt to complete a graph using SDP
Unfortunately completion number is not actually an SDP problem! Specifically, the problem of "remove as many negative eigenvalues as possible" is not convex because we care about the number, not the magnitude.
So instead what we do is add a low dimensional positive semidefinite term in a random subspace, and minimize its trace (which is the sum of the eigenvalues in that subspace).
This could actually add negative eigenvalues in the wrong places, but only a little bit (for reasons I don't fully understand)
That's why we currently force the whole thing to be a little "better" by adding an epsilon.
Also the thing fails sometimes because the optimization is too hard. We now have two dimensions of making it easier: reducing epsilon and increasing k.
'''
for trial in range(num_trials):
    # Random k-dimensional subspace
    V = np.random.randn(n, k)
    V, _ = np.linalg.qr(V)  # orthonormalize to n x k

    Y = cp.Variable((n, n), symmetric=True)
    Alpha = cp.Variable((k, k), PSD=True)

    constraints = [Y - eps*np.eye(n) +V @ Alpha @ V.T >> 0]

    for i in range(n):
        for j in range(n):
            if A_G[i, j] != 0:
                constraints.append(Y[i, j] == A_G[i, j])

    prob = cp.Problem(cp.Minimize(cp.trace(Alpha)), constraints)
    prob.solve()

    if prob.status == "optimal":
        obj = cp.trace(Alpha).value
        eigvals = np.linalg.eigvalsh(Y.value)
        num_neg = np.sum(eigvals < -1e-5) # When it was 6 I was getting a ton. My guess is the computation isn't that precise.

        print(f"Trial {trial}: trace(Alpha) = {obj:.6f}, neg eigenvalues: {num_neg}")
        neg_eigenvalues.append(num_neg)
        if obj < best_obj:
            best_obj = obj
            best_result = Y.value
print((neg_eigenvalues))
print("\nBest solution:")
eigvals = np.linalg.eigvalsh(best_result)
print("Eigenvalues:", np.sort(eigvals))
print("Negative eigenvalues:", np.sum(eigvals < -1e-5))

In [ ]:
import cvxpy as cp
import numpy as np
from itertools import combinations
# b cycle graphs with one negative edge each (no solution, k=1)

k = 4
b = 3
maximal_cliques = []
bad_edges =[]
for c in range(b):
    offset = c * k
    for i in range(k):
        maximal_cliques.append([offset + i, offset + (i + 1) % k])
    bad_edges.append([offset, offset + 1])
print(maximal_cliques)
print(bad_edges)
A_G = (generate_AG_bad(k * b, maximal_cliques, bad_edges=bad_edges))

n = A_G.shape[0]
k = 1  # number of allowed negative eigenvalues; completion number is the smallest k
num_trials = 100  # number of random subspaces to try

best_result = None
best_obj = np.inf
eps = 1e-4  # small shift to push near-zero eigenvalues positive
neg_eigenvalues = []
for trial in range(num_trials):
    # Random k-dimensional subspace
    V = np.random.randn(n, k)
    V, _ = np.linalg.qr(V)  # orthonormalize to n x k

    Y = cp.Variable((n, n), symmetric=True)
    Alpha = cp.Variable((k, k), PSD=True)

    constraints = [Y - eps*np.eye(n) +V @ Alpha @ V.T >> 0]

    for i in range(n):
        for j in range(n):
            if A_G[i, j] != 0:
                constraints.append(Y[i, j] == A_G[i, j])

    prob = cp.Problem(cp.Minimize(cp.trace(Alpha)), constraints)
    prob.solve()

    if prob.status == "optimal":
        obj = cp.trace(Alpha).value
        eigvals = np.linalg.eigvalsh(Y.value)
        num_neg = np.sum(eigvals < -1e-5) # When it was 6 I was getting a ton. My guess is the computation isn't that precise.

        print(f"Trial {trial}: trace(Alpha) = {obj:.6f}, neg eigenvalues: {num_neg}")
        neg_eigenvalues.append(num_neg)
        if obj < best_obj:
            best_obj = obj
            best_result = Y.value
            print(Y.value)
print((neg_eigenvalues))
print("\nBest solution:")
eigvals = np.linalg.eigvalsh(best_result)
print("Eigenvalues:", np.sort(eigvals))
print("Negative eigenvalues:", np.sum(eigvals < -1e-5))

In [ ]:
import cvxpy as cp
import numpy as np
from itertools import combinations
from finding_completion.tiling import tile_top_left_4x4
# confirming functionality using known bad matrix (counterexample from paper)

k = 4
b = 3
A_G = np.array(tile_top_left_4x4(3))
n = A_G.shape[0]
k = 4  # number of allowed negative eigenvalues; completion number is the smallest k
num_trials = 100  # number of random subspaces to try

best_result = None
best_obj = np.inf
eps = 1e-6  # small shift to push near-zero eigenvalues positive
neg_eigenvalues = []
for trial in range(num_trials):
    if (trial % 10 == 0):
        print("trial " + str(trial))
    # Random k-dimensional subspace
    V = np.random.randn(n, k)
    V, _ = np.linalg.qr(V)  # orthonormalize to n x k

    Y = cp.Variable((n, n), symmetric=True)
    Alpha = cp.Variable((k, k), PSD=True)

    constraints = [Y - eps*np.eye(n) + V @ Alpha @ V.T >> 0]

    for i in range(n):
        for j in range(n):
            if A_G[i, j] != None:
                constraints.append(Y[i, j] == A_G[i, j])

    prob = cp.Problem(cp.Minimize(cp.trace(Alpha)), constraints)
    prob.solve()

    if prob.status == "optimal":
        obj = cp.trace(Alpha).value
        eigvals = np.linalg.eigvalsh(Y.value)
        num_neg = np.sum(eigvals < -1e-5)

        print(f"Trial {trial}: trace(Alpha) = {obj:.6f}, neg eigenvalues: {num_neg}")
        neg_eigenvalues.append(num_neg)
        if obj < best_obj:
            best_obj = obj
            best_result = Y.value
            print(Y.value)
print((neg_eigenvalues))
print("\nBest solution:")
eigvals = np.linalg.eigvalsh(best_result)
print("Eigenvalues:", np.sort(eigvals))
print("Negative eigenvalues:", np.sum(eigvals < -1e-5))
# Apparently the PSD solver can't handle this as written. My guess is that this version scales in difficulty to roughly the number of *constraints* and this has a ton.
# Claude's brute forcey solver, on the other hand, can do these types of problems easily but struggles when there are too many options.